In [ ]:
import sys
sys.version

# Function Definitions

In [ ]:
#%matplotlib ipympl
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

import torch
import torch.optim as optim
#from torch_topological.nn import CubicalComplex
from cubical_complex import CubicalComplex # modified to allow V-construction
import gudhi as gd

#device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = 'cpu'
print('Device', device)

def pers_loss(D1, D2, pow=2, remove_longest_D1=True, remove_longest_D2=False):
    """
    Compute the persistence loss between two persistence diagrams D1 and D2.

    Parameters:
    D1, D2: torch.Tensor
        Persistence diagrams of shape [N, 2].
    pow: int, optional
        Power to which the differences are raised (default is 2).
    remove_longest_D1: bool, optional
        Whether to remove the longest persistence interval from D1 (default is True).
    remove_longest_D2: bool, optional
        Whether to remove the longest persistence interval from D2 (default is False).

    Returns:
    torch.Tensor
        The computed persistence loss.
    """
    pers1 = torch.diff(D1, dim=1).reshape(-1)
    pers1, _ = torch.sort(pers1, dim=0)
    if remove_longest_D1:
        pers1 = pers1[:-1]

    pers2 = torch.diff(D2, dim=1).reshape(-1)
    pers2, _ = torch.sort(pers2, dim=0)
    if remove_longest_D2:
        pers2 = pers2[:-1]

    if len(pers1) > len(pers2):
        p = len(pers1) - len(pers2)
        loss = (pers1[:p]).abs().pow(pow).sum()
        if len(pers2) > 0:
            loss += (pers1[p:] - pers2).abs().pow(pow).sum()
    else:
        p = len(pers2) - len(pers1)
        loss = (pers2[:p]).abs().pow(pow).sum()
        if len(pers1) > 0:
            loss += (pers2[p:] - pers1).abs().pow(pow).sum()

    return loss

from IPython.display import display, clear_output
def plot_loss(losses,ax=None):
    if ax is None:
        plt.figure(figsize=(8, 5))
        ax = fig.add_subplot(1,1,1)
    ax.clear()
    for key in losses.keys():
        ax.plot(losses[key], label=key)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_yscale('log')
    ax.legend()
    ax.grid(True)

def plot_gallery(images, title="", n_col=5, n_row=5, cmap=plt.cm.gray, axs=None):
    if axs is None:
        fig,axs = plt.subplots(n_row, n_col, figsize=(2. * n_col, 2.26 * n_row))
        plt.subplots_adjust(0.01, 0.05, 0.99, 0.93, 0.04, 0.)
        plt.suptitle(title, size=16)
    for i, comp in enumerate(images[:(n_col*n_row)]):
        vmax = max(comp.max(), -comp.min())
        axs[i//n_col,i%n_col].imshow(comp, cmap=cmap,
                   interpolation='nearest',
                   vmin=-vmax, vmax=vmax)
        axs[i//n_col,i%n_col].set_xticks(())
        axs[i//n_col,i%n_col].set_yticks(())

import gudhi
def plot_PD(images, n_col=5, n_row=5, axs=None, PHmode = "V", superlevel=False):
    if axs is None:
        fig,axs = plt.subplots(n_row, n_col, figsize=(2. * n_col, 2.26 * n_row))
    for i, comp in enumerate(images[:(n_col*n_row)]):
        sign = -1 if superlevel else 1
        if PHmode=="V":
            cubical_complex = gudhi.CubicalComplex(vertices=sign*comp)
        else:
            cubical_complex = gudhi.CubicalComplex(top_dimensional_cells=sign*comp)
        pd = cubical_complex.persistence()
        ax = axs[i//n_col,i%n_col]
        ax.clear()
        gudhi.plot_persistence_diagram(pd,axes=ax,legend=False,fontsize=4)
        ax.set_xticks(())
        ax.set_yticks(())


In [3]:
def update_V(X, W, V, k):
    WtW = W.T @ W
    cach = -W.T @ X + WtW @ V
    for i in range(V.shape[0]):
        C = cach[i] - WtW[i, i] * V[i]
        vi = torch.as_tensor(sparse_opt(-C.to('cpu').detach().numpy(), k), device=device)
        cach = cach + torch.outer(WtW[:,i],vi - V[i])
        V[i] = vi
    return V

def sparse_opt(b, k, epsilon=1e-10):
    if k<=0:
        return(b)
    permutation = np.argsort(b)[::-1]
    a = b[permutation]
    m = len(b)
    x = np.zeros(m, dtype=np.float64)

    y = np.zeros_like(x)
    bot = np.int64(np.ceil(k * k))
    if bot > m or k==1:
        y[permutation[0]]=1
        return y

    a_squared = a**2
    cumsum_a_squared = np.cumsum(a_squared)
    cumsum_a = np.cumsum(a)

    P = np.arange(1, m + 1, dtype=np.float64)

    numerator = P * cumsum_a_squared - cumsum_a**2
    denominator = P - k**2
    valid_mask = (denominator > 0) & (numerator > 0)
    #print(numerator,denominator,valid_mask)
    lambda_values = np.zeros_like(P)+epsilon
    mu_values = np.zeros_like(P)+epsilon
    lambda_values[valid_mask] = -np.sqrt(numerator[valid_mask] / denominator[valid_mask])
    mu_values[valid_mask] = -cumsum_a[valid_mask] / P[valid_mask] - k * lambda_values[valid_mask] / P[valid_mask]

    p_candidates = np.arange(np.int64(np.ceil(k * k)), m + 1)
    a_p_candidates = a[p_candidates - 1]
    mu_p_candidates = mu_values[p_candidates - 1]
    valid_p = np.where(a_p_candidates < -mu_p_candidates)[0]
    if len(valid_p)==0:
        pstar = m
    else:
        pstar = p_candidates[valid_p[0]]-1
    lam_pstar = lambda_values[pstar - 1]
    #print( lambda_values,mu_values,pstar)
    mu_pstar = mu_values[pstar - 1]
    x[:pstar] = -(a[:pstar] + mu_pstar) / lam_pstar

    y[permutation] = x
    return y

def sparsity_score(v):
    n = len(v.ravel())
    return((np.sqrt(n)-v.abs().sum()/(v**2).sum().sqrt())/(np.sqrt(n)-1))


In [4]:
def sparse_opt_hoyer(x, L1, L2=1, max_iter=100):
    dim = len(x)
    s = x + (L1 - np.sum(x)) / dim
    Z = set()

    for j in range(max_iter):
        # Step 3(a): Compute m_i
        m = np.zeros(dim)
        for i in range(dim):
            if i not in Z:
                m[i] = L1 / (dim - len(Z))

        # Step 3(b): Compute s
        alpha = 0
        sm = s - m
        a = np.sum(sm ** 2)
        b = 2 * np.dot(m, sm)
        c = np.sum(m ** 2) - L2 ** 2

        if a != 0:
            alpha = (-b + np.sqrt(b ** 2 - 4 * a * c)) / (2 * a)
        s = m + alpha * sm

        # Step 3(c): Check non-negativity
        if np.all(s >= 0):
            return s

        # Step 3(d): Update Z and set s_i = 0 for i in Z
        for i in range(dim):
            if s[i] < 0:
                Z.add(i)
                s[i] = 0

        # Step 3(e): Update s_i
        c = (np.sum(s) - L1) / (dim - len(Z))
        for i in range(dim):
            if i not in Z:
                s[i] -= c
    return(np.maximum(s,0))

In [ ]:
# test sparsity
b = np.array([0,1,3,2,4])
b = b/np.linalg.norm(b)
print(b)

k=2
v=sparse_opt(b, k)
print(v,v.sum(),np.linalg.norm(v),sparsity_score(torch.tensor(v)))
v=sparse_opt_hoyer(b, k)
print(v,v.sum(),np.linalg.norm(v),sparsity_score(torch.tensor(v)))


In [6]:
# synthetic image
import random

def create_ichimatsu_pattern(num_samples=100,image_shape = (36, 36),pat=None,pat_step=3,min_pat=10,max_pat=30,binarize=False, seed=42):
    random.seed(seed)
    X=[]
    for _ in range(num_samples):
        x = np.zeros(image_shape)
        for _ in range(random.randint(min_pat, max_pat)):
            cx = pat_step*random.randint(0, (image_shape[0]-pat.shape[0])//pat_step)
            cy = pat_step*random.randint(0, (image_shape[1]-pat.shape[1])//pat_step)
            x[cx:(cx+pat.shape[0]),cy:(cy+pat.shape[1])] += pat
            #print(cx,cy,x)
            if binarize:
                x=x>0
        X.append(x)
    return(np.array(X).astype(np.float64))


## top-NMF

In [ ]:
# face dataset
from sklearn.datasets import fetch_olivetti_faces
faces, _ = fetch_olivetti_faces(return_X_y=True, shuffle=True)
n_samples, n_features = faces.shape
image_shape = (64, 64)
print(faces.shape)


## data preparation

In [ ]:
## dataset of shape (n_samples, n_features)
#X = np.array([ [1,0,0,1,1] ]) # 1d
#X = np.array([ [[1,0,0],[1,0,1],[0,0,1]] ]) # 2d

# pat=np.zeros((9,9))
# pat[:,:3]=1
# pat[:3,:]=1

pat=np.zeros((6,6))
pat[0:2,0:2]=1
pat[4:6,4:6]=1

#X = create_ichimatsu_pattern(num_samples=40,image_shape = (18, 18),pat=pat,pat_step=3,min_pat=2,max_pat=8)
#X = create_ichimatsu_pattern(num_samples=1,image_shape = (18, 18),pat=pat,pat_step=3,min_pat=3,max_pat=3)
#X = create_ichimatsu_pattern(num_samples=40,image_shape = (18, 18),pat=pat,pat_step=3,min_pat=2,max_pat=8)
X = create_ichimatsu_pattern(num_samples=40,image_shape = (9, 9),pat=pat,pat_step=1,min_pat=2,max_pat=5, seed=41)

n_components = 32 # number of basis vectors
n_row, n_col = 4, 8 # display

# face dataset
# X = faces.reshape((len(faces),64,64))
# n_components = 34

### Start
n_samples = X.shape[0] # each row represents a sample
data_dims = X.shape[1:]
n_features = X[0].size

# Initialize with random nonnegative values
W = np.abs(np.random.normal(size=(n_samples, n_components)))
V = np.abs(np.random.normal(size=(n_components, n_features))) # each row represents a basis vector
W /= np.linalg.norm(X)

print(X.shape,W.shape,V.shape)


plot_gallery(X,title="original",n_row=n_row, n_col = n_col)


## optimisation setting

In [38]:
# fit
lr = 0.05 # learning rate for gradient descent  (Adam: 0.1, SGD: 5)
target_sparsity = None # set to None, if you are not using
epsilon = 1e-10 # for numerical stability
PHmode = "V" # T or V construction
superlevel = True

## specify target PH (or set to None for sparsity)
target_PH = [torch.tensor([[]], dtype=torch.float, device=device, requires_grad=False),torch.tensor([[]], dtype=torch.float, device=device, requires_grad=False)]

gd_iter = 1 # gradient descent update per iteration
mu_iter = 0 # multiplicative update per iteration
W_iter = 1 # W update per multiplicative update iteration
start_epoch_sparsify =0 # from this epoch, start sprase_opt
start_epoch_topological=100 # from this epoch, start loss_PH optimisation

lambda_apx = 1.0 # (we do not have to use for mu) reconstruction loss in gradient descent
lambda_spa_V = 0.0 # (we do not have to use for mu)
lambda_spa_W = 0.001 # (we do not have to use for mu)
lambda_top = 0.5 # 0.001 # top-NMF   # around 0.001
weight_decay = 0.01

init_optimizer = True

normalise = (target_sparsity is None) # divide by l2 after each iteration

show_plots = ["loss","basis","PH"] # ["loss","basis","PH"]

## filtration
cpx = CubicalComplex(superlevel=superlevel,dim=len(data_dims),mode=PHmode)

##
n_iterations = 2000
PH_dims = [0,1] # which dim to look at in PH
tol = 1e-4 # early stopping
tol_count = 1000 # early stopping
disp_interval = 10

#loss_fn = torch.nn.MSELoss() # approximation error
loss_fn = torch.nn.HuberLoss() # approximation error


## optimisation iteration

In [ ]:
##
X = torch.as_tensor(X.reshape(n_samples,-1), dtype=torch.float, device=device)
W = torch.tensor(W, dtype=torch.float, device=device, requires_grad=True)
V = torch.tensor(V, dtype=torch.float, device=device, requires_grad=True)
Vns = []
if normalise:
    with torch.no_grad():
        V /= (torch.norm(V, p=2, dim=1, keepdim=True)+epsilon)

## initialise optimiser
if init_optimizer:
    opt = optim.AdamW([W, V], lr=lr, weight_decay=weight_decay) # SGD, Adam, AdamW(weight decay for sparsity), or NAdam?
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.2, patience=1000)

## optimisation iteration
progress = tqdm(range(n_iterations))
prev_loss = np.inf
count = 0
losses = {"PH":[], "approx":[], "sparse_W":[], "sparse_V":[], "lr":[]}

if target_sparsity is not None and target_sparsity > 0:
    target_L1 = np.sqrt(V.shape[1]) - target_sparsity * (np.sqrt(V.shape[1]) - 1)
else:
    target_L1 = 0

# prepare figures
if "loss" in show_plots:
    fig_L,axs_L = plt.subplots(1,1,figsize=(8,5))
    disp_L = display(fig_L,display_id=True)
if "basis" in show_plots:
    fig_V,axs_V = plt.subplots(n_row,n_col, figsize=(2. * n_col, 2.26 * n_row))
    disp_V = display(fig_V,display_id=True)
if "PH" in show_plots:
    fig_P,axs_P = plt.subplots(n_row,n_col, figsize=(2. * n_col, 2.26 * n_row))
    disp_P = display(fig_P,display_id=True)

## main iteration
for epoch in progress:
    # multiplicative update
    with torch.no_grad():
        for _ in range(mu_iter):
            #update V
            if target_sparsity is None or epoch<start_epoch_sparsify:
                W_TX = W.T @ X
                W_TWV = W.T @ W @ V + epsilon
                V *= W_TX / W_TWV
            else:
                update_V(X, W, V, target_L1) # update "V" with target sparsity

            #update W
            for _ in range(W_iter):
                XV_T = X @ V.T
                WVV_T = W @ V @ V.T + epsilon
                W *= XV_T / WVV_T

    # gradient descent update
    for _ in range(gd_iter):
        # compute PD of rows of V
        loss_PH=torch.tensor(0., device=device)
        if target_PH is None:
            loss_PH -= n_components
        loss_spa_V=torch.tensor(0., device=device)
        loss_spa_W=torch.tensor(0., device=device)
        sp_score=torch.tensor(0., device=device)
        for j in range(n_components):
            v=V[j]
            # PH sparsity
            if lambda_top>0:
                diags = cpx(v.reshape(data_dims))
                if target_PH is None:
                    # PH sparsity
                    PH = torch.cat([diags[dim].diagram for dim in PH_dims])
                    pers1 = torch.diff(PH, dim=1).reshape(-1)
                    l1sq = pers1.sum()**2
                    l2sq = pers1.pow(2).sum()
                    loss_PH += (l1sq/(l2sq+epsilon))
                else:
                    # target PH
                    if epoch>=start_epoch_topological:
                        for dim in PH_dims:
                            if dim==0:
                                remove_longest_D1=True
                            else:
                                remove_longest_D1=False
                            loss_PH += pers_loss(diags[dim].diagram, target_PH[dim],remove_longest_D1=remove_longest_D1)

            # sparsity
            #loss_spa += (sparsity_score(v)-target_sparsity)**2
            #loss_spa += (1-sparsity_score(v))
            loss_spa_V += (v.abs().sum())**2/(v**2).sum()
            sp_score += sparsity_score(v)

        #loss_spa_W = (W.abs().sum())**2/(W**2).sum()
        ## row-wise
        #loss_spa_W = torch.sum(torch.sum(torch.abs(W), dim=1)**2/torch.sum(W**2, dim=1))
        for j in range(n_samples):
            w=W[j]
            loss_spa_W += (w.abs().sum())**2/(w**2).sum()

        loss_spa_V /= n_components
        loss_spa_W /= n_components
        loss_PH /= n_components
        sp_score /= n_components
        loss_apx = loss_fn(torch.mm(W, V),X)

        # gradient descent
        loss = lambda_top * loss_PH + lambda_spa_V * loss_spa_V + lambda_spa_W * loss_spa_W +  lambda_apx * loss_apx
        opt.zero_grad()
        loss.backward()
        #vv = V.clone().detach()
        opt.step()
        scheduler.step(loss)
        losses["PH"].append(loss_PH.item())
        losses["approx"].append(loss_apx.item())
        losses["sparse_V"].append(loss_spa_V.item())
        losses["sparse_W"].append(loss_spa_W.item())
        losses["lr"].append(scheduler.get_last_lr()[0])

    # Enforce nonnegativity by clamping
    with torch.no_grad():
        W.clamp_(min=0)
        V.clamp_(min=0)
        if normalise:
            V /= (torch.norm(V, p=2, dim=1, keepdim=True)+epsilon)

    # display
    progress.set_postfix(loss=f'PH: {loss_PH.item():.08f},sparsity: {sp_score.item():.08f},approx: {loss_apx.item():.08f}')
    if epoch%disp_interval==0:
        Vns.append(V.detach().cpu().numpy())
        #clear_output(wait=True)

        # for key in losses.keys():
        #     line[key].set_xdata(range(len(losses[key])))
        #     line[key].set_ydata(losses[key])
        if "loss" in show_plots:
            plot_loss(losses,ax=axs_L)
            disp_L.update(fig_L)
        if "basis" in show_plots:
            plot_gallery(Vns[-1].reshape(-1,*data_dims),title="basis",n_row=n_row, n_col = n_col, axs=axs_V)
            disp_V.update(fig_V)
        if "PH" in show_plots:
            plot_PD(Vns[-1].reshape(-1,*data_dims),n_row=n_row, n_col = n_col, axs=axs_P,PHmode = PHmode, superlevel=superlevel)
            disp_P.update(fig_P)
        #plt.show()
        #display(plt.gcf())
        disp_interval = int(1.2*disp_interval) # increase interval

    # early stopping
    current = lambda_top * loss_PH + lambda_spa_V * loss_spa_V + lambda_spa_W * loss_spa_W + loss_apx
    if (prev_loss-current)<tol:
      count += 1
      if count > tol_count:
        break
    else:
      prev_loss = current.item()
      count = 0

# check norm
plt.close('all')
print("target L1/L2 ratio",target_L1)
#print(np.linalg.norm(V.detach(),axis=1))

In [ ]:
idx=13
v=V.reshape(-1,*data_dims)[idx]
diags = cpx(v)
plt.imshow(v.detach())
pers_loss(diags[0].diagram, target_PH[0]),pers_loss(diags[1].diagram, target_PH[1],remove_longest_D1=False)

In [ ]:
cubical_complex = gudhi.CubicalComplex(vertices=-v.detach().reshape(data_dims))
pd = cubical_complex.persistence()
gudhi.plot_persistence_diagram(pd,fontsize=4)

In [ ]:
diags

In [ ]:
PH = torch.cat([cpx(v.reshape(data_dims))[dim].diagram for dim in PH_dims])
pers1 = torch.diff(PH, dim=1).reshape(-1)
pers1

## results

In [ ]:
## get results
Xn = X.detach().cpu().numpy()
Wn = W.detach().cpu().numpy()
Vn = V.detach().cpu().numpy()

num = 5
fig,axs = plt.subplots(2,num,figsize=(15,5))
for i in range(num):
    if len(data_dims)==1:
        sns.barplot(X[i],ax=axs[0,i])
        sns.barplot( (Wn@Vn)[i],ax=axs[1,i])
    else:
        axs[0,i].imshow(X[i].reshape(data_dims))
        axs[1,i].imshow((Wn@Vn)[i].reshape(data_dims))
        axs[0,i].axis("off")
        axs[0,i].set_title("original")
        axs[1,i].axis("off")
        axs[1,i].set_title("reconstructed")
plot_gallery(Vn.reshape(-1,*data_dims),title="basis",n_row=n_row, n_col = n_col)
plot_PD(Vn.reshape(-1,*data_dims),n_row=n_row, n_col = n_col, PHmode=PHmode)


In [ ]:
U=torch.tensor(V[0]).reshape(data_dims)
cpx(U),U


## conventional

In [ ]:
from sklearn import decomposition

n_components = 34 # number of basis vectors
l1_ratio = 0.3
model = decomposition.NMF(n_components=n_components, init='nndsvdar',solver='mu',alpha_W=0.001, alpha_H='same', l1_ratio=l1_ratio)
model.fit(faces)
V=model.components_
result = model.inverse_transform(model.transform(faces))

plot_gallery(result,title="reconstructed",n_row=5, n_col = 5)

In [ ]:
plot_gallery(V,title="basis",n_row=5, n_col = 5)

In [ ]:
import importlib
import sparse_nmf
importlib.reload(sparse_nmf)

spar = 0.5
maxiter = 2000
V,W = sparse_nmf.sparse_nmf(faces.T, n_components, maxiter, spar)
plot_gallery(V.T,title="basis",n_row=5, n_col = 5)